# Claim Extraction — roBERTa + spaCy

This notebook focuses on the first stage of the misinformation verification pipeline: claim extraction.  
The goal is to take raw claims from the LIAR dataset and convert them into structured `(subject, relation, object)` triples that can later be checked against external knowledge sources and used in probabilistic reasoning.

For example:

`"NASA confirmed water on Mars"`  
becomes:

- subject → NASA  
- relation → confirmed  
- object → water on Mars  

The notebook also generates confidence scores for each extracted claim structure, which will be used in later stages of the project.

**Primary models and tools used:**
- `roBERTa-base` from HuggingFace for contextual language understanding
- `dslim/bert-base-NER` for named entity recognition (NER)
- spaCy `en_core_web_sm` for dependency parsing and relation extraction


## 1. Setup

In [3]:
!pip3 install transformers datasets torch spacy pandas tqdm --quiet
!python -m spacy download en_core_web_sm --quiet


[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: pip3 install --upgrade pip
zsh:1: command not found: python


In [4]:
import json
import os
import re
from pathlib import Path
import warnings
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import spacy
import torch
from tqdm import tqdm
from transformers import pipeline

warnings.filterwarnings("ignore")
torch.manual_seed(42)
print("Libraries loaded.")

Libraries loaded.


## 2. Load LIAR Dataset

LIAR is tab-separated with no header row. Only `id`, `label`, `statement`, `subject`, and `speaker` are needed here.

In [5]:
LIAR_COLUMNS = [
    "id", "label", "statement", "subject", "speaker",
    "job_title", "state", "party",
    "barely_true_counts", "false_counts", "half_true_counts",
    "mostly_true_counts", "pants_fire_counts", "context",
]

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "LIAR_dataset").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "LIAR_dataset"
KEEP_COLS = ["id", "label", "statement", "subject", "speaker"]

def load_split(filename):
    df = pd.read_csv(
        DATA_DIR / filename,
        sep="\t", header=None, names=LIAR_COLUMNS, dtype=str,
    )
    return df[KEEP_COLS].copy()

df_train = load_split("train.tsv")
df_val   = load_split("valid.tsv")
df_test  = load_split("test.tsv")

for df in [df_train, df_val, df_test]:
    df.fillna("", inplace=True)

print(f"Train: {len(df_train):,} | Val: {len(df_val):,} | Test: {len(df_test):,}")
print(f"\nLabel distribution (train):\n{df_train['label'].value_counts()}")
df_train.head()

Train: 10,240 | Val: 1,284 | Test: 1,267

Label distribution (train):
label
half-true      2114
false          1995
mostly-true    1962
true           1676
barely-true    1654
pants-fire      839
Name: count, dtype: int64


,id,label,statement,subject,speaker
0,2635.json,false,Says the Annies List political group supports ...,abortion,dwayne-bohac
1,10540.json,half-true,When did the decline of coal start? It started...,"energy,history,job-accomplishments",scott-surovell
2,324.json,mostly-true,"Hillary Clinton agrees with John McCain ""by vo...",foreign-policy,barack-obama
3,1123.json,false,Health care reform legislation is likely to ma...,health-care,blog-posting
4,9028.json,half-true,The economic turnaround started at the end of ...,"economy,jobs",charlie-crist


## 3. Preprocessing

Light normalization only — collapse whitespace, strip non-ASCII. RoBERTa handles casing and punctuation internally, so no lowercasing.

In [6]:
def preprocess_claim(text: str) -> str:
    if not isinstance(text, str) or not text.strip():
        return ""
    text = text.strip()
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"[^\x00-\x7F]+", " ", text)
    return re.sub(r"\s+", " ", text).strip()

for df in [df_train, df_val, df_test]:
    df["statement"] = df["statement"].apply(preprocess_claim)

initial_count = len(df_train)
df_train = df_train[df_train["statement"].str.len() > 0].reset_index(drop=True)
print(f"Removed {initial_count - len(df_train)} empty records from train split")

print("\nSample cleaned statements:")
for stmt in df_train["statement"].head(5):
    print(f"  - {stmt}")

Removed 0 empty records from train split

Sample cleaned statements:
  - Says the Annies List political group supports third-trimester abortions on demand.
  - When did the decline of coal start? It started when natural gas took off that started to begin in (President George W.) Bushs administration.
  - Hillary Clinton agrees with John McCain "by voting to give George Bush the benefit of the doubt on Iran."
  - Health care reform legislation is likely to mandate free sex change surgeries.
  - The economic turnaround started at the end of my term.


## 4. Named Entity Recognition

We use RoBERTa to read each claim and identify the important words,
labeling them as people, organizations, or locations.

- "Barack Obama" → Person  
- "California" → Location  
- "NASA" → Organization

These labeled words become the subjects and objects in our triples.

In [7]:
NER_MODEL_NAME = "Jean-Baptiste/roberta-large-ner-english"

print(f"Loading {NER_MODEL_NAME}...")
ner_pipeline = pipeline(
    task="ner",
    model=NER_MODEL_NAME,
    aggregation_strategy="simple",
)
print("NER pipeline ready.")

Loading Jean-Baptiste/roberta-large-ner-english...


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

NER pipeline ready.


In [8]:
# Sanity check on known examples
TEST_CLAIMS = [
    "NASA confirmed water on Mars in 2015.",
    "Barack Obama raised taxes on the middle class.",
    "The unemployment rate under President Trump hit a 50-year low.",
    "California has the highest income tax rate in the United States.",
]

print("NER validation:\n")
for claim in TEST_CLAIMS:
    entities = ner_pipeline(claim)
    print(f"Claim: {claim}")
    for ent in entities:
        print(f"  [{ent['entity_group']}] '{ent['word']}' (score={ent['score']:.3f})")
    print()

NER validation:

Claim: NASA confirmed water on Mars in 2015.
  [ORG] ' NASA' (score=0.998)
  [LOC] ' Mars' (score=0.999)

Claim: Barack Obama raised taxes on the middle class.
  [PER] ' Barack Obama' (score=0.998)

Claim: The unemployment rate under President Trump hit a 50-year low.
  [PER] ' Trump' (score=0.991)

Claim: California has the highest income tax rate in the United States.
  [LOC] ' California' (score=1.000)
  [LOC] ' United States.' (score=0.993)



## 5. Relation Extraction — spaCy Dependency Parse

Uses spaCy’s dependency tree to find the `(subject, relation, object)` from each sentence. RoBERTa handles accurate entity detection, while spaCy identifies how those entities are connected in the sentence structure. 


In [9]:
print("Loading spaCy parser...")
nlp = spacy.load("en_core_web_sm")
print("spaCy parser ready.")

Loading spaCy parser...
spaCy parser ready.


In [10]:
def extract_relation_from_dep_tree(doc) -> Tuple[str, str, str]:
    """
    Extract (subject, relation, object) from a spaCy dependency parse.
    Looks for root verb, then walks children for nsubj and dobj/attr/pobj.
    Falls back to rightmost noun chunk if no direct object is found.
    """
    subject = ""
    relation = ""
    obj = ""

    noun_chunks = {chunk.root.i: chunk.text for chunk in doc.noun_chunks}

    root_token = next((t for t in doc if t.dep_ == "ROOT"), None)
    if root_token is None:
        return subject, relation, obj

    aux_tokens = [t.text for t in root_token.children if t.dep_ in ("aux", "auxpass")]
    relation = (" ".join(aux_tokens) + " " if aux_tokens else "") + root_token.lemma_

    for child in root_token.children:
        if child.dep_ in ("nsubj", "nsubjpass"):
            subject = noun_chunks.get(child.i, child.text)
            break

    for child in root_token.children:
        if child.dep_ == "dobj":
            obj = noun_chunks.get(child.i, child.text)
            break
        elif child.dep_ == "attr":
            obj = noun_chunks.get(child.i, child.text)
            break

    if not obj:
        for child in root_token.children:
            if child.dep_ == "prep":
                for grandchild in child.children:
                    if grandchild.dep_ == "pobj":
                        obj = child.text + " " + noun_chunks.get(grandchild.i, grandchild.text)
                        break

    if not obj:
        post_verb = [c.text for c in doc.noun_chunks if c.root.i > root_token.i]
        if post_verb:
            obj = post_verb[-1]

    return subject.strip(), relation.strip(), obj.strip()


print("Relation extraction validation:\n")
for claim in TEST_CLAIMS:
    doc = nlp(claim)
    subj, rel, obj = extract_relation_from_dep_tree(doc)
    print(f"Claim: {claim}")
    print(f"  Subject:  {subj}")
    print(f"  Relation: {rel}")
    print(f"  Object:   {obj}")
    print()

Relation extraction validation:

Claim: NASA confirmed water on Mars in 2015.
  Subject:  NASA
  Relation: confirm
  Object:   water

Claim: Barack Obama raised taxes on the middle class.
  Subject:  Barack Obama
  Relation: raise
  Object:   taxes

Claim: The unemployment rate under President Trump hit a 50-year low.
  Subject:  The unemployment rate
  Relation: hit
  Object:   a 50-year low

Claim: California has the highest income tax rate in the United States.
  Subject:  California
  Relation: have
  Object:   the highest income tax rate



## 6. NER-Grounded Triple Refinement

Checks whether the subject or object extracted by spaCy matches an entity detected by RoBERTa. If matched, the pipeline replaces it with RoBERTa’s cleaner entity span and attaches the entity type label (`PER`, `ORG`, `LOC`, etc.).


In [11]:
def refine_triple_with_ner(
    subject: str,
    relation: str,
    obj: str,
    ner_entities: List[Dict],
) -> Dict:
    """
    Replace syntactic subject/object spans with overlapping NER spans where available.
    Subject priority: PER > ORG > LOC > MISC
    Object priority:  LOC > ORG > MISC > PER
    """
    subject_type = "UNKNOWN"
    object_type  = "UNKNOWN"

    subject_priority = {"PER": 0, "ORG": 1, "LOC": 2, "MISC": 3}
    object_priority  = {"LOC": 0, "ORG": 1, "MISC": 2, "PER": 3}

    subject_candidates = []
    object_candidates  = []

    for ent in ner_entities:
        ent_text  = ent["word"].strip()
        ent_group = ent["entity_group"]
        ent_score = ent["score"]

        if ent_text.lower() in subject.lower() or subject.lower() in ent_text.lower():
            subject_candidates.append((ent_text, ent_group, ent_score,
                                        subject_priority.get(ent_group, 99)))
        if ent_text.lower() in obj.lower() or obj.lower() in ent_text.lower():
            object_candidates.append((ent_text, ent_group, ent_score,
                                       object_priority.get(ent_group, 99)))

    if subject_candidates:
        best = sorted(subject_candidates, key=lambda x: x[3])[0]
        subject, subject_type = best[0], best[1]

    if object_candidates:
        best = sorted(object_candidates, key=lambda x: x[3])[0]
        obj, object_type = best[0], best[1]

    return {
        "subject":      subject,
        "subject_type": subject_type,
        "relation":     relation,
        "object":       obj,
        "object_type":  object_type,
    }

## 7. Confidence Scoring

Gives each extracted triple a reliability score based on three things:

* `0.5` → how confident RoBERTa is about the detected entities
* `0.3` → whether the subject, relation, and object were properly extracted
* `0.2` → how many parts were successfully identified as real entities instead of `UNKNOWN`

Higher scores usually mean the extracted information is more reliable.


In [12]:
def compute_extraction_confidence(
    triple: Dict,
    ner_entities: List[Dict],
) -> float:
    """Returns a heuristic confidence score in [0.0, 1.0]."""
    ner_confidence = (
        sum(e["score"] for e in ner_entities) / len(ner_entities)
        if ner_entities else 0.0
    )
    components = [triple["subject"], triple["relation"], triple["object"]]
    structural_score = sum(1 for c in components if c.strip()) / 3.0

    grounded = sum(1 for k in ("subject_type", "object_type")
                   if triple.get(k, "UNKNOWN") != "UNKNOWN")
    grounding_score = grounded / 2.0

    confidence = (
        0.5 * ner_confidence +
        0.3 * structural_score +
        0.2 * grounding_score
    )
    return round(min(max(confidence, 0.0), 1.0), 4)

## 8. Extraction Pipeline

Combines all components — entity detection, sentence parsing, relationship extraction, date detection, and confidence scoring — into one complete processing pipeline.

In [13]:
DATE_PATTERNS = [
    r"\b(19|20)\d{2}\b",
    r"\b(?:January|February|March|April|May|June|July|August|September"
    r"|October|November|December)\s+\d{1,2},?\s+\d{4}\b",
    r"\b\d{1,2}/\d{1,2}/\d{2,4}\b",
]


def extract_date_metadata(text: str) -> Optional[str]:
    """Return the first date/year expression found in the claim, or None."""
    for pattern in DATE_PATTERNS:
        match = re.search(pattern, text)
        if match:
            return match.group(0)
    return None


def _empty_result(text: str) -> Dict:
    return {
        "raw_claim": text, "subject": "", "subject_type": "",
        "relation": "", "object": "", "object_type": "",
        "date": None, "confidence": 0.0, "entities": [],
    }


def extract_claim(
    text: str,
    ner_pipe,
    spacy_nlp,
    speaker: str = "",
) -> Dict:
    """
    Full extraction for a single claim string.
    Returns: subject, subject_type, relation, object, object_type,
             date, confidence, raw_claim, entities.
    """
    if not text or not text.strip():
        return _empty_result(text)

    try:
        ner_entities = ner_pipe(text)
    except Exception:
        ner_entities = []

    doc = spacy_nlp(text)
    subj, rel, obj = extract_relation_from_dep_tree(doc)
    triple = refine_triple_with_ner(subj, rel, obj, ner_entities)

    if not triple["subject"] and speaker:
        triple["subject"] = speaker
        triple["subject_type"] = "PER"

    date_meta  = extract_date_metadata(text)
    confidence = compute_extraction_confidence(triple, ner_entities)

    return {
        "raw_claim":    text,
        "subject":      triple["subject"],
        "subject_type": triple["subject_type"],
        "relation":     triple["relation"],
        "object":       triple["object"],
        "object_type":  triple["object_type"],
        "date":         date_meta,
        "confidence":   confidence,
        "entities":     [{"word": e["word"], "type": e["entity_group"],
                          "score": round(e["score"], 4)} for e in ner_entities],
    }


print("Full pipeline test:\n")
for claim in TEST_CLAIMS:
    result = extract_claim(claim, ner_pipeline, nlp)
    print(f"Claim: {result['raw_claim']}")
    print(f"  Triple:     ({result['subject']} [{result['subject_type']}], "
          f"{result['relation']}, "
          f"{result['object']} [{result['object_type']}])")
    print(f"  Date:       {result['date']}")
    print(f"  Confidence: {result['confidence']}")
    print(f"  Entities:   {result['entities']}")
    print()

Full pipeline test:

Claim: NASA confirmed water on Mars in 2015.
  Triple:     (NASA [ORG], confirm, water [UNKNOWN])
  Date:       2015
  Confidence: 0.899399995803833
  Entities:   [{'word': ' NASA', 'type': 'ORG', 'score': np.float32(0.9985)}, {'word': ' Mars', 'type': 'LOC', 'score': np.float32(0.999)}]

Claim: Barack Obama raised taxes on the middle class.
  Triple:     (Barack Obama [PER], raise, taxes [UNKNOWN])
  Date:       None
  Confidence: 0.8988999724388123
  Entities:   [{'word': ' Barack Obama', 'type': 'PER', 'score': np.float32(0.9979)}]

Claim: The unemployment rate under President Trump hit a 50-year low.
  Triple:     (The unemployment rate [UNKNOWN], hit, a 50-year low [UNKNOWN])
  Date:       None
  Confidence: 0.7957000136375427
  Entities:   [{'word': ' Trump', 'type': 'PER', 'score': np.float32(0.9913)}]

Claim: California has the highest income tax rate in the United States.
  Triple:     (California [LOC], have, the highest income tax rate [UNKNOWN])
  Date:

## 9. Batch Processing

NER and spaCy both support batched inference. Processing in chunks of `batch_size` to keep memory manageable.

In [14]:
def process_dataframe_batch(
    df: pd.DataFrame,
    ner_pipe,
    spacy_nlp,
    batch_size: int = 64,
    max_records: Optional[int] = None,
) -> List[Dict]:
    """
    Batch-process a LIAR DataFrame through the extraction pipeline.
    Reduce batch_size if running into OOM on CPU.
    """
    subset     = df if max_records is None else df.head(max_records)
    statements = subset["statement"].tolist()
    speakers   = subset["speaker"].tolist()
    labels     = subset["label"].tolist()

    results = []

    for start in tqdm(range(0, len(statements), batch_size), desc="Extracting claims"):
        batch_texts    = statements[start : start + batch_size]
        batch_speakers = speakers[start : start + batch_size]
        batch_labels   = labels[start : start + batch_size]

        try:
            batch_ner = ner_pipe(batch_texts, batch_size=batch_size)
        except Exception as e:
            print(f"NER batch error at index {start}: {e}")
            batch_ner = [[] for _ in batch_texts]

        batch_docs = list(spacy_nlp.pipe(batch_texts))

        for text, speaker, label, ner_ents, doc in zip(
            batch_texts, batch_speakers, batch_labels, batch_ner, batch_docs
        ):
            subj, rel, obj = extract_relation_from_dep_tree(doc)
            triple = refine_triple_with_ner(subj, rel, obj, ner_ents)

            if not triple["subject"] and speaker:
                triple["subject"] = speaker
                triple["subject_type"] = "PER"

            date_meta  = extract_date_metadata(text)
            confidence = compute_extraction_confidence(triple, ner_ents)

            results.append({
                "raw_claim":    text,
                "speaker":      speaker,
                "label":        label,
                "subject":      triple["subject"],
                "subject_type": triple["subject_type"],
                "relation":     triple["relation"],
                "object":       triple["object"],
                "object_type":  triple["object_type"],
                "date":         date_meta,
                "confidence":   confidence,
                "entities":     json.dumps([{
                    "word":  e["word"],
                    "type":  e["entity_group"],
                    "score": round(float(e["score"]), 4),
                } for e in ner_ents]),
            })

    return results


print("Batch processor defined.")

Batch processor defined.


In [15]:
# max_records=None for a full run (~10k records, several hours on CPU)
# Use 200-500 for local CPU demo, 2000-5000 on Colab free GPU
MAX_RECORDS = 500

print(f"Processing {MAX_RECORDS} training records...")
extraction_results = process_dataframe_batch(
    df_train,
    ner_pipeline,
    nlp,
    batch_size=32,
    max_records=MAX_RECORDS,
)

df_extracted = pd.DataFrame(extraction_results)
print(f"\nExtraction complete. {len(df_extracted)} records processed.")
print(f"Columns: {list(df_extracted.columns)}")

Processing 500 training records...


Extracting claims: 100%|██████████| 16/16 [00:07<00:00,  2.22it/s]


Extraction complete. 500 records processed.
Columns: ['raw_claim', 'speaker', 'label', 'subject', 'subject_type', 'relation', 'object', 'object_type', 'date', 'confidence', 'entities']


## 10. Output Inspection

In [16]:
total = len(df_extracted)

subject_fill_rate  = (df_extracted["subject"]  != "").sum() / total
relation_fill_rate = (df_extracted["relation"] != "").sum() / total
object_fill_rate   = (df_extracted["object"]   != "").sum() / total
full_triple_rate   = (
    (df_extracted["subject"]  != "") &
    (df_extracted["relation"] != "") &
    (df_extracted["object"]   != "")
).sum() / total

print("Extraction coverage:")
print(f"  Subject fill rate:     {subject_fill_rate:.1%}")
print(f"  Relation fill rate:    {relation_fill_rate:.1%}")
print(f"  Object fill rate:      {object_fill_rate:.1%}")
print(f"  Full triple fill rate: {full_triple_rate:.1%}")
print(f"\nConfidence score stats:")
print(df_extracted["confidence"].describe().round(4))

Extraction coverage:
  Subject fill rate:     100.0%
  Relation fill rate:    100.0%
  Object fill rate:      99.2%
  Full triple fill rate: 99.2%

Confidence score stats:
count    500.0000
mean       0.7735
std        0.2345
min        0.2000
25%        0.7974
50%        0.8963
75%        0.8999
max        0.9999
Name: confidence, dtype: float64


In [17]:
display_cols = ["raw_claim", "subject", "subject_type",
                "relation", "object", "object_type", "date", "confidence"]

print("Sample extracted triples (first 10):\n")

with pd.option_context(
    "display.max_colwidth", None,
    "display.width", 250,
    "display.expand_frame_repr", False
):
    display(df_extracted[display_cols].head(10))

Sample extracted triples (first 10):



,raw_claim,subject,subject_type,relation,object,object_type,date,confidence
0,Says the Annies List political group supports third-trimester abortions on demand.,Annies List,ORG,say,demand,UNKNOWN,NaN,0.8949
1,When did the decline of coal start? It started when natural gas took off that started to begin in (President George W.) Bushs administration.,the decline,UNKNOWN,do,George W.) Bush,PER,NaN,0.8983
2,"Hillary Clinton agrees with John McCain ""by voting to give George Bush the benefit of the doubt on Iran.""",Hillary Clinton,PER,agree,John McCain,PER,NaN,0.9998
3,Health care reform legislation is likely to mandate free sex change surgeries.,Health care reform legislation,UNKNOWN,be,free sex change surgeries,UNKNOWN,NaN,0.3000
4,The economic turnaround started at the end of my term.,The economic turnaround,UNKNOWN,start,at the end,UNKNOWN,NaN,0.3000
5,The Chicago Bears have had more starting quarterbacks in the last 10 years than the total number of tenured (UW) faculty fired during the last two decades.,Chicago Bears,ORG,have have,more starting quarterbacks,UNKNOWN,NaN,0.8934
6,Jim Dunnam has not lived in the district he represents for years now.,Jim Dunnam,PER,has live,in the district,UNKNOWN,NaN,0.8999
7,"I'm the only person on this stage who has worked actively just last year passing, along with Russ Feingold, some of the toughest ethics reform since Watergate.",Russ Feingold,PER,be,the only person,UNKNOWN,NaN,0.7779
8,"However, it took $19.5 million in Oregon Lottery funds for the Port of Newport to eventually land the new NOAA Marine Operations Center-Pacific.",it,UNKNOWN,take,million,UNKNOWN,NaN,0.6284
9,Says GOP primary opponents Glenn Grothman and Joe Leibham cast a compromise vote that cost $788 million in higher electricity costs.,Glenn Grothman,PER,say,higher electricity costs,UNKNOWN,NaN,0.8998


In [24]:
HIGH_CONF_THRESHOLD = 0.75

df_high_conf = df_extracted[
    df_extracted["confidence"] >= HIGH_CONF_THRESHOLD
]

print(
    f"High-confidence triples "
    f"(>= {HIGH_CONF_THRESHOLD}): "
    f"{len(df_high_conf)} / {total} "
    f"({len(df_high_conf)/total:.1%})\n"
)

with pd.option_context(
    "display.max_colwidth", None,
    "display.width", 250,
    "display.expand_frame_repr", False
):
    display(
        df_high_conf[display_cols]
        .head(8)
        .style
        .hide(axis="index")
    )

High-confidence triples (>= 0.75): 396 / 500 (79.2%)



raw_claim,subject,subject_type,relation,object,object_type,date,confidence
Says the Annies List political group supports third-trimester abortions on demand.,Annies List,ORG,say,demand,UNKNOWN,nan,0.894900
When did the decline of coal start? It started when natural gas took off that started to begin in (President George W.) Bushs administration.,the decline,UNKNOWN,do,George W.) Bush,PER,nan,0.898300
"Hillary Clinton agrees with John McCain ""by voting to give George Bush the benefit of the doubt on Iran.""",Hillary Clinton,PER,agree,John McCain,PER,nan,0.999800
The Chicago Bears have had more starting quarterbacks in the last 10 years than the total number of tenured (UW) faculty fired during the last two decades.,Chicago Bears,ORG,have have,more starting quarterbacks,UNKNOWN,nan,0.893400
Jim Dunnam has not lived in the district he represents for years now.,Jim Dunnam,PER,has live,in the district,UNKNOWN,nan,0.899900
"I'm the only person on this stage who has worked actively just last year passing, along with Russ Feingold, some of the toughest ethics reform since Watergate.",Russ Feingold,PER,be,the only person,UNKNOWN,nan,0.777900
Says GOP primary opponents Glenn Grothman and Joe Leibham cast a compromise vote that cost $788 million in higher electricity costs.,Glenn Grothman,PER,say,higher electricity costs,UNKNOWN,nan,0.899800
"For the first time in history, the share of the national popular vote margin is smaller than the Latino vote margin.",the share,UNKNOWN,be,For the first time,UNKNOWN,nan,0.799600


In [25]:
LOW_CONF_THRESHOLD = 0.35

df_low_conf = df_extracted[
    df_extracted["confidence"] < LOW_CONF_THRESHOLD
]

print(
    f"Low-confidence triples "
    f"(< {LOW_CONF_THRESHOLD}): "
    f"{len(df_low_conf)} / {total}\n"
)

low_conf_cols = [
    "raw_claim",
    "subject",
    "relation",
    "object",
    "confidence"
]

with pd.option_context(
    "display.max_colwidth", None,
    "display.width", 250,
    "display.expand_frame_repr", False
):
    display(
        df_low_conf[low_conf_cols]
        .head(8)
        .style
        .hide(axis="index")
    )

Low-confidence triples (< 0.35): 82 / 500



raw_claim,subject,relation,object,confidence
Health care reform legislation is likely to mandate free sex change surgeries.,Health care reform legislation,be,free sex change surgeries,0.300000
The economic turnaround started at the end of my term.,The economic turnaround,start,at the end,0.300000
The economy bled $24 billion due to the government shutdown.,The economy,bleed,billion,0.300000
Youth unemployment in minority communities is about 40 to 45 percent.,Youth unemployment,be,about 40 to 45 percent,0.300000
"If you look at states that are right to work, they constantly do not have budget deficits and they have very good business climates.",they,do have,budget deficits,0.300000
We cut business taxes so today 70 percent of our businesses dont pay a business tax.,We,cut,business taxes,0.300000
We have a federal government that thinks they have the authority to regulate our toilet seats.,We,have,a federal government,0.300000
We have a director of homeland security who cannot use and will not use the term 'terrorist attack' but instead substitutes 'man-made disaster.',We,have,a director,0.300000


## 11. Save Outputs

- **CSV** (full): all records including partial extractions, for inspection and analysis.
- **JSON** (filtered): records where subject and relation are both non-empty, for downstream ingestion.

In [20]:
OUTPUT_DIR = "outputs/phase2"
os.makedirs(OUTPUT_DIR, exist_ok=True)

csv_path = os.path.join(OUTPUT_DIR, "extracted_triples_full.csv")
df_extracted.to_csv(csv_path, index=False)
print(f"Full results saved:     {csv_path}")

df_valid = df_extracted[
    (df_extracted["subject"] != "") &
    (df_extracted["relation"] != "")
].copy()

json_path = os.path.join(OUTPUT_DIR, "extracted_triples_filtered.json")
df_valid.to_json(json_path, orient="records", indent=2)
print(f"Filtered results saved: {json_path}  ({len(df_valid)} records)")

print("\nSample JSON record:")
sample = df_valid.head(1).to_dict(orient="records")[0]
print(json.dumps(sample, indent=2, default=str))

Full results saved:     outputs/phase2/extracted_triples_full.csv
Filtered results saved: outputs/phase2/extracted_triples_filtered.json  (500 records)

Sample JSON record:
{
  "raw_claim": "Says the Annies List political group supports third-trimester abortions on demand.",
  "speaker": "dwayne-bohac",
  "label": "false",
  "subject": "Annies List",
  "subject_type": "ORG",
  "relation": "say",
  "object": "demand",
  "object_type": "UNKNOWN",
  "date": NaN,
  "confidence": 0.8949000239372253,
  "entities": "[{\"word\": \" Annies List\", \"type\": \"ORG\", \"score\": 0.9897}]"
}


## 12. Evaluation

Since LIAR has no official correct triples, the pipeline is evaluated using:

- **Manual review** — randomly inspect extracted triples for correctness  
- **Coverage check** — measure how often subject, relation, and object are successfully extracted  
- **Entity distribution check** — verify whether detected entity types look realistic overall  

Unusual patterns usually indicate extraction errors.

In [21]:
# Method B: Coverage
print("=== Coverage Analysis ===")
print(f"  Total records:        {total}")
print(f"  Subject non-empty:    {subject_fill_rate:.1%}")
print(f"  Relation non-empty:   {relation_fill_rate:.1%}")
print(f"  Object non-empty:     {object_fill_rate:.1%}")
print(f"  Full triple (S+R+O):  {full_triple_rate:.1%}")
print(f"  With date metadata:   {(df_extracted['date'].notna()).mean():.1%}")

# Method C: Entity type distribution
print("\n=== Entity Type Distribution ===")
print("Subject types:")
print(df_extracted["subject_type"].value_counts().to_string())
print("\nObject types:")
print(df_extracted["object_type"].value_counts().to_string())

# Confidence by label — should be roughly label-independent
print("\n=== Mean Confidence by Label ===")
label_map = {
    "0": "pants-fire", "1": "false", "2": "barely-true",
    "3": "half-true",  "4": "mostly-true", "5": "true"
}
df_extracted["label_name"] = df_extracted["label"].map(label_map)
print(df_extracted.groupby("label_name")["confidence"].mean().round(4))
print("\nNote: large variance across labels may indicate extraction quality")
print("is correlated with claim complexity rather than veracity.")

=== Coverage Analysis ===
  Total records:        500
  Subject non-empty:    100.0%
  Relation non-empty:   100.0%
  Object non-empty:     99.2%
  Full triple (S+R+O):  99.2%
  With date metadata:   7.4%

=== Entity Type Distribution ===
Subject types:
subject_type
UNKNOWN    211
PER        162
LOC         53
MISC        39
ORG         35

Object types:
object_type
UNKNOWN    401
LOC         33
MISC        32
PER         20
ORG         14

=== Mean Confidence by Label ===
Series([], Name: confidence, dtype: float64)

Note: large variance across labels may indicate extraction quality
is correlated with claim complexity rather than veracity.


A stratified sample means the data is split into groups first, then samples are taken from each group separately.

Here, the groups are confidence quartiles:

- Q1 → lowest confidence triples  
- Q2 → lower-middle confidence  
- Q3 → upper-middle confidence  
- Q4 → highest confidence triples  

The code randomly picks 5 examples from each group so you can manually inspect extraction quality across all confidence levels instead of only seeing high-confidence results.

In [26]:
# Method A: Stratified sample for manual review

print("=== Stratified Sample (5 per confidence quartile) ===\n")

df_extracted["conf_quartile"] = pd.qcut(
    df_extracted["confidence"],
    q=4,
    labels=[
        "Q1 (low)",
        "Q2",
        "Q3",
        "Q4 (high)"
    ]
)

stratified_sample = (
    df_extracted
    .groupby("conf_quartile", observed=True)
    .apply(
        lambda g: g.sample(
            min(5, len(g)),
            random_state=42
        ),
        include_groups=False
    )
    .reset_index(level=0)
    .reset_index(drop=True)
)

sample_cols = [
    "conf_quartile",
    "raw_claim",
    "subject",
    "relation",
    "object",
    "confidence"
]

with pd.option_context(
    "display.max_colwidth", None,
    "display.width", 250,
    "display.expand_frame_repr", False
):
    display(
        stratified_sample[sample_cols]
        .style
        .hide(axis="index")
    )

=== Stratified Sample (5 per confidence quartile) ===



conf_quartile,raw_claim,subject,relation,object,confidence
Q1 (low),We built a new prison every 10 days between 1990 and 2005 to keep up with our mass incarceration explosion of nonviolent offenders.,We,build,a new prison,0.300000
Q1 (low),Says judges get better benefits at a lower cost than everybody else in the state.,chris-christie,say,the state,0.400000
Q1 (low),Im the only one whos fought against developers draining the Everglades!,I,m,one,0.792800
Q1 (low),We spend less on defense today as % of GDP than at any time since Pearl Harbor.,We,spend,less,0.792500
Q1 (low),The United States imprisons more than any nation in the world.,United States,imprison,the world,0.780000
Q2,This year in Congress (Connie Mack IV) has missed almost half of his votes.,This year,has miss,almost half,0.798800
Q2,"Says the man who rushed the stage at him in Dayton, Ohio, had chatter about ISIS, or with ISIS in his social media posts.",ISIS,say,his social media posts,0.864900
Q2,John wasn't this raging populist four years ago when he ran for president.,this,was rag,president,0.799100
Q2,Firearms homicides are down about 40 percent since Texas passed concealed-gun permit law.,homicides,be,concealed-gun permit law,0.799800
Q2,"It has been estimated that nearly 40 percent of all guns sold in America are sold by private, unlicensed sellers either online or through gun shows.",It,has been estimate,gun shows,0.800000


## 13. Known Limitations

- Pronouns like “he” or “they” may not link correctly to the real person.  
- Only one main relationship is extracted per claim.  
- Short or messy political statements can confuse spaCy’s parser.  
- RoBERTa may miss unusual or informal entity names.  
- Confidence scores are approximate, not exact probabilities.  

**Possible improvements:** better relation extraction models, coreference resolution, political-domain NER fine-tuning, and improved confidence calibration.

In [23]:
print("=" * 50)
print("Claim Extraction Complete")
print("=" * 50)
print(f"  Records processed:        {total}")
print(f"  Full triples extracted:   {int(full_triple_rate * total)} ({full_triple_rate:.1%})")
print(f"  High-confidence (>=0.75): {len(df_high_conf)} ({len(df_high_conf)/total:.1%})")
print(f"  Output (CSV):             {csv_path}")
print(f"  Output (JSON):            {json_path}")
print()
print("Fields in filtered JSON: subject, subject_type, relation,")
print("  object, object_type, confidence, date")

Claim Extraction Complete
  Records processed:        500
  Full triples extracted:   496 (99.2%)
  High-confidence (>=0.75): 396 (79.2%)
  Output (CSV):             outputs/phase2/extracted_triples_full.csv
  Output (JSON):            outputs/phase2/extracted_triples_filtered.json

Fields in filtered JSON: subject, subject_type, relation,
  object, object_type, confidence, date
